# PSRT Bearing Demo

Set `DATA_DIR` to a folder of CWRU `.mat` files, then run the cells top to bottom.

In [ ]:
from pathlib import Path

from psrt_bearing.classify import train_evaluate
from psrt_bearing.data import load_cwru_mat, recording_id
from psrt_bearing.embedding import estimate_tau, farthest_point_sample, plot_embedding, takens_embedding
from psrt_bearing.featurize import DiskFeatureCache, featurize_window
from psrt_bearing.window import labeled_windows

DATA_DIR = Path("../data/cwru")
ARTIFACT_DIR = Path("../artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

In [ ]:
records = []
for mat_path in sorted(DATA_DIR.glob("*.mat")):
    signal, label = load_cwru_mat(mat_path)
    windows = labeled_windows(signal, label=label, length=4096)
    if windows:
        records.append((recording_id(mat_path), label, windows[0][0]))

healthy = next(item for item in records if item[1] == "healthy")
faulty = next(item for item in records if item[1] == "faulty")
healthy, faulty

In [ ]:
for source, label, window in (healthy, faulty):
    tau = estimate_tau(window)
    cloud = takens_embedding(window, dimension=3, delay=tau)
    sampled = farthest_point_sample(cloud, max_points=256)
    plot_embedding(sampled, ARTIFACT_DIR / f"{label}_embedding.png", title=f"{source}: {label}")

In [ ]:
cache = DiskFeatureCache(ARTIFACT_DIR / "feature-cache")
features = []
labels = []
groups = []
for mat_path in sorted(DATA_DIR.glob("*.mat"))[:8]:
    signal, label = load_cwru_mat(mat_path)
    source = recording_id(mat_path)
    for window, _ in labeled_windows(signal, label=label, length=4096)[:2]:
        result = featurize_window(
            window,
            radii=None,
            betti_keys=((1, 2), (2, 3), (2, 4)),
            embedding_dim=3,
            delay=estimate_tau(window),
            max_points=16,
            max_dim=2,
            max_subset_card=4,
            method="psrt-pairs",
            normalize=True,
            cache=cache,
        )
        features.append(result.values)
        labels.append(label)
        groups.append(source)

len(features), len(cache)

In [ ]:
train_evaluate(features, labels, groups=groups)